# 02 — Pull Universe Trades

For every event in `data/event_universe.csv`, enumerate all sub-market tickers
from `data/markets_metadata.csv` and pull historical trades via
`get_historical_trades()` (same function as `pull_data.ipynb`).

**Skip-if-exists:** already-pulled CSVs are not re-downloaded.  
**Re-run safe:** running this notebook again is a no-op for completed pulls.

In [1]:
import requests
import pandas as pd
import time
import os

BASE_URL = 'https://api.elections.kalshi.com/trade-api/v2'
BASE_DIR = os.path.abspath('..')
DATA_DIR = os.path.join(BASE_DIR, 'data')

KEEP_COLS        = ['trade_id', 'ticker', 'created_time', 'count_fp',
                    'yes_price_dollars', 'no_price_dollars', 'taker_side']
SLEEP_BETWEEN    = 0.25    # seconds between pulls

print(f'DATA_DIR     : {DATA_DIR}')
print(f'SLEEP_BETWEEN: {SLEEP_BETWEEN}s')

DATA_DIR     : /Users/josephsemelroth/Desktop/QSS 20 Final Project/data
SLEEP_BETWEEN: 0.25s


In [2]:
def get_historical_trades(ticker, max_pages=None, verbose=False):
    """Identical to pull_data.ipynb — do not modify."""
    all_trades = []
    cursor     = None
    page       = 0

    while True:
        params = {'ticker': ticker, 'limit': 1000}
        if cursor:
            params['cursor'] = cursor

        r = requests.get(f'{BASE_URL}/historical/trades', params=params)
        if r.status_code != 200:
            if verbose:
                print(f'  Error {r.status_code}: {r.text[:200]}')
            break

        data   = r.json()
        trades = data.get('trades', [])
        all_trades.extend(trades)
        page  += 1

        cursor = data.get('cursor')
        if not cursor or len(trades) == 0:
            break
        if max_pages and page >= max_pages:
            break

    if verbose:
        print(f'  done: {len(all_trades):,} trades for {ticker}')
    return pd.DataFrame(all_trades)


print('get_historical_trades defined.')

get_historical_trades defined.


In [3]:
univ = pd.read_csv(os.path.join(DATA_DIR, 'event_universe.csv'))
meta = pd.read_csv(os.path.join(DATA_DIR, 'markets_metadata.csv'), low_memory=False)

print(f'event_universe:   {len(univ):,} events')
print(f'markets_metadata: {len(meta):,} sub-markets')
print()

# Filter metadata to only events in universe (case-insensitive)
event_tickers_upper = set(univ['event_ticker'].str.upper())
meta_upper          = meta.copy()
meta_upper['_ev_upper'] = meta_upper['event_ticker'].str.upper()
meta_in_univ = meta_upper[meta_upper['_ev_upper'].isin(event_tickers_upper)].copy()

print(f'Sub-markets in universe events: {len(meta_in_univ):,}')
print()

# Warn about any events missing from metadata
meta_events_upper = set(meta['event_ticker'].str.upper())
missing           = event_tickers_upper - meta_events_upper
if missing:
    print(f'WARNING: {len(missing)} events in universe not found in metadata:')
    for ev in sorted(missing):
        print(f'  {ev}')
else:
    print('All universe events found in metadata.')

event_universe:   182 events
markets_metadata: 3,393 sub-markets

Sub-markets in universe events: 2,255

All universe events found in metadata.


In [4]:
# Pre-scan disk to count what's already there
n_already = sum(
    1 for tk in meta_in_univ['ticker']
    if os.path.exists(os.path.join(DATA_DIR, f'{tk.lower()}_trades.csv'))
)
n_todo = len(meta_in_univ) - n_already

print(f'Sub-markets already on disk: {n_already:,}')
print(f'Sub-markets to pull:         {n_todo:,}')
print()
if n_todo > 0:
    est_min = n_todo * SLEEP_BETWEEN / 60
    print(f'Estimated pull time at {SLEEP_BETWEEN}s/market: ~{est_min:.0f} min')
else:
    print('All sub-markets already on disk — run will be a no-op.')

Sub-markets already on disk: 2,255
Sub-markets to pull:         0

All sub-markets already on disk — run will be a no-op.


In [5]:
print(f'Pulling trades for {len(meta_in_univ):,} sub-markets across {len(univ):,} events ...')
print()

summary_rows = []

for ev_ticker, ev_group in meta_in_univ.groupby('event_ticker'):
    tickers_in_event = ev_group['ticker'].tolist()
    n_found = n_pulled = n_skipped = n_zero = n_error = 0

    for tk in tickers_in_event:
        n_found  += 1
        out_path  = os.path.join(DATA_DIR, f'{tk.lower()}_trades.csv')

        if os.path.exists(out_path):
            n_skipped += 1
            continue

        try:
            df = get_historical_trades(tk, verbose=False)
        except Exception as exc:
            print(f'  ERROR pulling {tk}: {exc}')
            n_error += 1
            continue

        if df.empty:
            pd.DataFrame(columns=KEEP_COLS).to_csv(out_path, index=False)
            n_zero += 1
        else:
            df[[c for c in KEEP_COLS if c in df.columns]].to_csv(out_path, index=False)
            n_pulled += 1

        time.sleep(SLEEP_BETWEEN)

    summary_rows.append({
        'event_ticker':  ev_ticker,
        'n_sub_markets': n_found,
        'n_pulled':      n_pulled,
        'n_skipped':     n_skipped,
        'n_zero_trade':  n_zero,
        'n_error':       n_error,
    })

    flag = '  !! ERROR' if n_error else ''
    print(f'  {ev_ticker:<38}  sub={n_found:>3}  '
          f'pulled={n_pulled:>3}  skipped={n_skipped:>3}  '
          f'zero={n_zero:>2}  err={n_error}{flag}')

print()
print('Pull loop complete.')

Pulling trades for 2,255 sub-markets across 182 events ...

  KXACAHOUSEVOTE-27JAN01                  sub= 10  pulled=  0  skipped= 10  zero= 0  err=0
  KXALCARAZCOACH-26                       sub= 12  pulled=  0  skipped= 12  zero= 0  err=0
  KXARGENTINAHOUSETHIRD-25OCT26-3         sub=  4  pulled=  0  skipped=  4  zero= 0  err=0
  KXARGENTINAMOSTGAIN-25OCT26             sub=  4  pulled=  0  skipped=  4  zero= 0  err=0
  KXARKCOACH-26                           sub= 19  pulled=  0  skipped= 19  zero= 0  err=0
  KXASIALEADEROUT-35                      sub= 10  pulled=  0  skipped= 10  zero= 0  err=0
  KXAUBCOACH-26                           sub= 27  pulled=  0  skipped= 27  zero= 0  err=0
  KXBUCHARESTMAYOR2ND-25DEC07-2           sub=  4  pulled=  0  skipped=  4  zero= 0  err=0
  KXBUCHARESTMAYOR3RD-25DEC07-3           sub=  5  pulled=  0  skipped=  5  zero= 0  err=0
  KXCABOUT-26MAR                          sub= 24  pulled=  0  skipped= 24  zero= 0  err=0
  KXCABOUT-29                 

In [6]:
summary = pd.DataFrame(summary_rows)

print('PULL SUMMARY')
print('=' * 82)
print(f'  {"Event":<38}  {"Sub":>4}  {"Pulled":>6}  {"Skipped":>7}  {"Zero":>5}  {"Err":>4}')
print('  ' + '-' * 78)
for _, r in summary.iterrows():
    print(f'  {r["event_ticker"]:<38}  {r["n_sub_markets"]:>4}  '
          f'{r["n_pulled"]:>6}  {r["n_skipped"]:>7}  '
          f'{r["n_zero_trade"]:>5}  {r["n_error"]:>4}')
print('  ' + '-' * 78)
print(f'  {"TOTAL":<38}  {summary["n_sub_markets"].sum():>4}  '
      f'{summary["n_pulled"].sum():>6}  {summary["n_skipped"].sum():>7}  '
      f'{summary["n_zero_trade"].sum():>5}  {summary["n_error"].sum():>4}')
print()
print(f'Total sub-markets in universe : {summary["n_sub_markets"].sum():,}')
print(f'Pulled this run               : {summary["n_pulled"].sum():,}')
print(f'Skipped (already on disk)     : {summary["n_skipped"].sum():,}')
print(f'Zero-trade (empty file saved) : {summary["n_zero_trade"].sum():,}')
print(f'Errors                        : {summary["n_error"].sum():,}')

if summary['n_error'].sum() > 0:
    print()
    print('WARNING: errors occurred — re-run notebook to retry failed tickers.')

PULL SUMMARY
  Event                                    Sub  Pulled  Skipped   Zero   Err
  ------------------------------------------------------------------------------
  KXACAHOUSEVOTE-27JAN01                    10       0       10      0     0
  KXALCARAZCOACH-26                         12       0       12      0     0
  KXARGENTINAHOUSETHIRD-25OCT26-3            4       0        4      0     0
  KXARGENTINAMOSTGAIN-25OCT26                4       0        4      0     0
  KXARKCOACH-26                             19       0       19      0     0
  KXASIALEADEROUT-35                        10       0       10      0     0
  KXAUBCOACH-26                             27       0       27      0     0
  KXBUCHARESTMAYOR2ND-25DEC07-2              4       0        4      0     0
  KXBUCHARESTMAYOR3RD-25DEC07-3              5       0        5      0     0
  KXCABOUT-26MAR                            24       0       24      0     0
  KXCABOUT-29                               26       0     